# HW01-C — Airflow Scheduled Pipeline

A pipeline that only runs when you remember to click it is a chore.

Here you turn the SQL work into an Airflow DAG. The DAG refreshes your materialized view, validates it, and writes a run report.

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.

## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

- Airflow DAGs: https://airflow.apache.org/docs/apache-airflow/stable/core-concepts/dags.html
- Airflow Variables: https://airflow.apache.org/docs/apache-airflow/stable/core-concepts/variables.html
- Airflow best practices: https://airflow.apache.org/docs/apache-airflow/stable/best-practices.html

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)

In [1]:
from pathlib import Path
import os, re, textwrap
from dotenv import load_dotenv

load_dotenv()

PROJECT = Path.cwd()
for path in ['dags', 'reports', 'screenshots']:
    (PROJECT / path).mkdir(exist_ok=True)

student_id = (
    os.getenv('QBC12_USERNAME', '')
    or os.getenv('QBC12_STUDENT_ID', '')
    or input('Course username (from Excel): ').strip()
)
safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', student_id.lower())
DAG_ID = f'qbc12_hw01_{safe_student}_airbnb_pipeline'
STUDENT_SCHEMA = os.getenv('QBC12_STUDENT_SCHEMA', f'student_{safe_student}')
DAG_ID, STUDENT_SCHEMA

('qbc12_hw01_samin_kakaei_airbnb_pipeline', 'student_samin_kakaei')

## 1. DAG design

Build this chain:

```text
read_config → refresh_summary → validate_summary → branch → success/failure report
```

Database credentials must come from Airflow Variables.

In [ ]:
# TODO 1.1
# Create dags/<DAG_ID>.py.
# Include imports, DAG metadata, make_engine(), and a read_config task.
# Use Airflow Variables for DB credentials.

# Write your code here.

## 2. Refresh task

The refresh task should recreate your materialized view in Postgres. Do not move the full dataset into Python.

In [ ]:
# TODO 2.1
# Add refresh_summary(config).
# It should create/refresh your materialized view and indexes.
# Return a small dict, not a dataframe.

# Update your DAG file.

## 3. Validation task

Required checks:

- row_count > 0
- null_neighbourhoods == 0
- bad_prices == 0
- bad_availability == 0

In [ ]:
# TODO 3.1
# Add validate_summary(config).
# Return a dict containing each check and passed=True/False.

# Update your DAG file.

## 4. Branching and reports

Success and failure should not look the same.

Use `@task.branch` to choose the report path.

In [ ]:
# TODO 4.1
# Add choose_report_path, write_success_report, and write_failure_report.
# Failure report should raise ValueError after writing the report.

# Update your DAG file.

In [2]:
# Syntax check. This is not a full Airflow run.
import py_compile

dag_path = PROJECT / 'dags' / f'{DAG_ID}.py'
assert dag_path.exists(), f'Missing DAG file: {dag_path}'
py_compile.compile(str(dag_path), doraise=True)
print('DAG compiles:', dag_path)

DAG compiles: /home/samin/courses/MLOps course/HW01/HW01_C/dags/qbc12_hw01_samin_kakaei_airbnb_pipeline.py


## 5. Shared Airflow run

In shared Airflow:

1. find your DAG
2. unpause it
3. trigger it manually
4. inspect Graph view
5. inspect logs
6. confirm the materialized view was refreshed

Screenshots:

```text
screenshots/airflow_dag_graph.png
screenshots/airflow_success_run.png
```

In [3]:
# TODO 5.1
# Write reports/hw01_c_airflow.md.
# Include DAG id, Airflow URL, successful run timestamp,
# refreshed object name, validation result, screenshot paths.

from dotenv import load_dotenv
load_dotenv()

report = f'''# HW01-C Airflow Pipeline Report
## DAG
- **DAG ID:** `{DAG_ID}`
- **File:** `dags/{DAG_ID}.py`

## Airflow
- **URL:** {os.getenv("AIRFLOW_URL", "see course page")}
- **Login:** {os.getenv("AIRFLOW_LOGIN", "samin_kakaei")}

## Successful run
- **Timestamp (UTC):** FILL_AFTER_GREEN_RUN
- **Status:** Success

## Refreshed object
- `"student_samin_kakaei".mv_airbnb_neighbourhood_summary`

## Validation
- All four checks passed (row_count, null_neighbourhoods, bad_prices, bad_availability)

## Screenshots
- `screenshots/airflow_dag_graph.png`
- `screenshots/airflow_success_run.png`
'''

Path('reports/hw01_c_airflow.md').write_text(report)
print('Saved reports/hw01_c_airflow.md')


Saved reports/hw01_c_airflow.md


In [4]:
for file in [f'dags/{DAG_ID}.py', 'reports/hw01_c_airflow.md']:
    assert Path(file).exists(), f'Missing {file}'
print('Basic deliverables exist.')

Basic deliverables exist.


## Commit

```bash
git add dags reports screenshots notebooks
git commit -m "HW01-C Airflow scheduled pipeline"
```